In [ ]:
! pip install pandas requests python-dotenv openpyxl xlrd jupyterlab

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached jupyterlab-4.5.7-py3-none-any.whl.metadata (16 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.18.2-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB

In [5]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import json

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")

print("API Key loaded:", "✓" if API_KEY else "✗ — cek file .env kamu")

API Key loaded: ✓


In [6]:
def fetch_eia(route, params):
    """Hit EIA API v2 dan kembalikan dataframe."""
    base_url = f"https://api.eia.gov/v2/{route}"
    params["api_key"] = API_KEY
    
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")
    
    data = response.json()
    records = data.get("response", {}).get("data", [])
    
    print(f"  → {len(records)} records diterima")
    return pd.DataFrame(records)

In [7]:
print("Mengambil data Net Generation...")

df_generation = fetch_eia(
    route="electricity/electric-power-operational-data",
    params={
        "frequency": "monthly",
        "data[]": "generation",
        "facets[location][]": "US",
        "facets[fueltypeid][]": ["COL", "NG", "NUC", "WND", "SUN", "HYC"],
        "start": "2022-01",
        "end": "2024-12",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000
    }
)

df_generation.head(3)

Mengambil data Net Generation...
  → 15 records diterima


,generation,total-consumption,consumption-for-eg,consumption-uto,total-consumption-btu,consumption-for-eg-btu,consumption-uto-btu,stocks,receipts,receipts-btu,cost,cost-per-btu,sulfur-content,ash-content,heat-content
alias,Utility Scale Electricity Net Generation,Consumption of Fuels for Electricity Generatio...,Consumption of Fuels for Electricity Generatio...,Consumption of Fuels for Useful Thermal Output...,Consumption of Fuels for Electricity Generatio...,Consumption of Fuels for Electricity Generatio...,Consumption of Fuels for Useful Thermal Output...,Stocks of Fuel (Physical Units),Receipts of Fuel (Physical Units),Receipts of Fuel (BTUs),Average Cost of Fuels (per Physical Unit),Average Cost of Fuels (per BTU),Average Sulfur Content of Consumed Fuel,Average Ash Content of Consumed Fuel,Average Heat Content of Consumed Fuels


In [8]:
print("Mengambil data Retail Sales & Price...")

df_retail = fetch_eia(
    route="electricity/retail-sales",
    params={
        "frequency": "monthly",
        "data[]": ["sales", "price", "revenue", "customers"],
        "facets[stateid][]": "US",
        "facets[sectorid][]": ["RES", "COM", "IND", "ALL"],
        "start": "2022-01",
        "end": "2024-12",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000
    }
)

df_retail.head(3)

Mengambil data Retail Sales & Price...
  → 4 records diterima


,revenue,sales,price,customers
alias,Revenue from Sales to Ultimate Customers,Megawatt-hours Sold to Ultimate Customers,Average Price of Electricity to Ultimate Custo...,Number of Ultimate Customers
units,million dollars,million kilowatt hours,cents per kilowatt-hour,number of customers


In [9]:
# Cek struktur data
print("=== Net Generation ===")
print("Kolom:", df_generation.columns.tolist())
print("Shape:", df_generation.shape)
print()
print("=== Retail Sales ===")
print("Kolom:", df_retail.columns.tolist())
print("Shape:", df_retail.shape)

# Simpan ke staging
os.makedirs("../data/staging", exist_ok=True)

df_generation.to_csv("../data/staging/eia_generation_2022_2024.csv", index=False)
df_retail.to_csv("../data/staging/eia_retail_2022_2024.csv", index=False)

print("\n✓ File berhasil disimpan di data/staging/")

=== Net Generation ===
Kolom: ['generation', 'total-consumption', 'consumption-for-eg', 'consumption-uto', 'total-consumption-btu', 'consumption-for-eg-btu', 'consumption-uto-btu', 'stocks', 'receipts', 'receipts-btu', 'cost', 'cost-per-btu', 'sulfur-content', 'ash-content', 'heat-content']
Shape: (1, 15)

=== Retail Sales ===
Kolom: ['revenue', 'sales', 'price', 'customers']
Shape: (2, 4)

✓ File berhasil disimpan di data/staging/
